# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We list all `RecordSet` objects with their `@id`, name, and description. For each record set, we show its fields (columns) and their IDs. You will need these IDs for precise field extraction and referencing.

In [ ]:
# Print all RecordSets, their @id, and fields info
recordset_list = list(dataset.metadata.record_sets)
print(f"Number of record sets: {len(recordset_list)}\n")
for rs in recordset_list:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print(f"  Fields (columns):")
    for field in rs.fields:
        print(f"    Field @id: {field.id}, Name: {field.name}, DataType: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

**Tip:** Replace the variable values in the code below with the correct `@id`s from your dataset overview.

For this FAIR^2 dataset, we will demonstrate using one of the available `RecordSet` `@id`s. Adjust `record_sets` and `record_set_id` as appropriate for the specific data table you wish to analyze.

In [ ]:
# Replace below with actual RecordSet @ids found in Data Overview
record_sets = []
# For demonstration, auto-discover all record set IDs for exploration
record_sets = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"Warning: Record set {record_set_id} is empty or unavailable.")
    else:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for RecordSet {record_set_id} with columns: {df.columns.tolist()}")

# Select a sample record set for EDA (if available)
if len(dataframes) > 0:
    record_set_id = list(dataframes.keys())[0]
    print(f"First 5 records from RecordSet: {record_set_id}")
    display(dataframes[record_set_id].head(5))
else:
    print("No record sets with loaded data found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates removal of outliers, data normalization, and grouping. Please adjust the field `@id`s for the numeric and grouping variables based on your data overview output above.

**Note:** For this example, we will attempt to select the first numeric column found. For thorough analysis, refer to your `fields` output and use the correct `@id` for domain-relevant variables.

In [ ]:
import numpy as np
if len(dataframes) > 0:
    df = dataframes[record_set_id]
    # Try to select the first numeric-type column
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if not numeric_field:
        # Fallback: try converting possible columns to numeric
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if df[col].notna().sum() > 0:
                    numeric_field = col
                    break
            except Exception:
                continue

    if numeric_field:
        print(f"Selected numeric field for analysis: {numeric_field}")
        # Filter records where numeric_field > threshold
        threshold = df[numeric_field].mean() if np.isfinite(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold} (mean value):")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field (pick first non-numeric)
        group_field = None
        for col in df.columns:
            if df[col].dtype == object and col != numeric_field:
                group_field = col
                break
        if group_field:
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print("Grouped means by categorical field:")
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field available for analysis in this record set.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we plot the distribution of the selected numeric field, and, if grouping is available, also a bar plot per group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if len(dataframes) > 0 and 'numeric_field' in locals() and numeric_field and numeric_field in df:
    # Histogram
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in RecordSet {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    # Show boxplot by group, if grouping variable found
    if 'group_field' in locals() and group_field and group_field in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Insufficient data for plotting.")

## 6. Conclusion
In this notebook, we loaded and explored the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset using the `mlcroissant` library. We inspected the metadata for available record sets and fields, loaded specific tables into pandas DataFrames, and applied basic exploratory analysis and visualization. This workflow demonstrates a reproducible and FAIR approach to dataset exploration in Python.

Further analyses can be performed by referencing individual fields and record sets using their `@id` as illustrated.

*Don't forget to cite the dataset as:*

> Kulka, M., Rodriguez Miera, S. and Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.